# Flood Forecasting for the River Iskar

This notebook demonstrates a machine learning–based simulation model to forecast potential flood events along the River Iskar (Bulgaria) using synthetic data. The model leverages meteorological and hydrological features and uses a regression approach to predict the river water level.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# For reproducibility
np.random.seed(42)

## Data Generation

We generate synthetic data simulating daily measurements for one year (2023). The features include precipitation and upstream flow, and the target variable is the river water level. A seasonal component and random noise are added to mimic real-world conditions.

In [ ]:
# Generate date range for one year
dates = pd.date_range(start='2023-01-01', end='2023-12-31')
n = len(dates)

# Create synthetic data
data = pd.DataFrame({
    'date': dates,
    # Precipitation with occasional heavy rains (in mm)
    'precipitation': np.clip(np.random.normal(loc=2, scale=3, size=n), 0, None),
    # Simulated upstream flow (in m³/s)
    'upstream_flow': np.random.normal(loc=100, scale=20, size=n)
})

# Generate target water level (in meters) using a synthetic relationship
data["water_level"] = (
    0.05 * data['precipitation'] + 
    0.03 * data['upstream_flow'] +
    np.sin(np.linspace(0, 3 * np.pi, n)) +   # seasonal component
    np.random.normal(scale=0.5, size=n)      # noise component
)

data.head()

## Data Preprocessing

To capture the temporal dependencies in the data, we create lag features. Then, we split the data into training and testing sets (keeping time order) and scale the features.

In [ ]:
# Create lag features (lag of 1 day)
data['precipitation_lag1'] = data['precipitation'].shift(1)
data['upstream_flow_lag1'] = data['upstream_flow'].shift(1)

# Drop the first row which contains NaN values because of lag features
data = data.dropna().reset_index(drop=True)

# Define features and the target variable
features = ['precipitation', 'upstream_flow', 'precipitation_lag1', 'upstream_flow_lag1']
target = 'water_level'

# Separate features and target
X = data[features]
y = data[target]

# Split the data into training and testing sets while maintaining time order
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

data.head()

## Model Training

We use a RandomForestRegressor as our forecasting model. The model is trained on the training dataset and its performance is evaluated on both the training and test sets.

In [ ]:
# Initialize and train the RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Evaluate model performance
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Train RMSE: {train_rmse:.3f}, R2: {train_r2:.3f}")
print(f"Test RMSE: {test_rmse:.3f}, R2: {test_r2:.3f}")

## Results Visualization

The following plot shows the observed water levels and the predicted water levels for the test set. A horizontal line represents a hypothetical flood threshold (e.g., 7 m), which can be used to identify potential flood risk conditions.

In [ ]:
# Combine dates with observed and predicted water levels for the test set
results = data.iloc[y_train.shape[0]:].copy()
results['predicted_water_level'] = y_test_pred

plt.figure(figsize=(14, 6))
plt.plot(results['date'], results[target], label='Observed Water Level')
plt.plot(results['date'], results['predicted_water_level'], label='Predicted Water Level', linestyle='--')
# Hypothetical flood threshold (e.g., water level >= 7 m)
plt.axhline(y=7, color='grey', linestyle=':', label='Flood Threshold (7 m)')
plt.xlabel('Date')
plt.ylabel('Water Level (m)')
plt.title('River Iskar Water Level Forecasting')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Simulation and Scenario Analysis

To further explore the model's response to extreme events, we simulate an extreme precipitation event. In this example, a precipitation spike is induced by multiplying the precipitation value by 5 on a selected day in the test set, and then the model's predicted water level is recalculated.

In [ ]:
# Create a copy of the test set to simulate the scenario
X_test_scenario = X_test.copy()

# Choose an index in the test set (example index 20) and simulate extreme precipitation
scenario_index = X_test_scenario.index[20]
X_test_scenario.iloc[20, X_test.columns.get_loc('precipitation')] *= 5  

# Scale the modified test set
X_test_scenario_scaled = scaler.transform(X_test_scenario)

# Predict with the modified data
y_test_scenario_pred = model.predict(X_test_scenario_scaled)

# Update scenario results
results_scenario = results.copy()
results_scenario.loc[results.index[20], 'predicted_water_level'] = y_test_scenario_pred[20]

plt.figure(figsize=(14, 6))
plt.plot(results_scenario['date'], results_scenario[target], label='Observed Water Level')
plt.plot(results_scenario['date'], results_scenario['predicted_water_level'], label='Predicted Water Level', linestyle='--')
plt.axhline(y=7, color='grey', linestyle=':', label='Flood Threshold (7 m)')
plt.xlabel('Date')
plt.ylabel('Water Level (m)')
plt.title('Scenario Analysis: Effect of Extreme Precipitation on Flood Forecasting')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we built a machine learning simulation model to forecast the water levels of the River Iskar using synthetic data. We generated data with a seasonal component and noise, engineered lag features, trained a Random Forest regressor, and visualized both the model predictions and the impact of an extreme precipitation scenario. For real-world applications, integrating actual hydrological and meteorological data is essential for reliable flood forecasting.